# V5: Stage2 Transformer Baseline  
### V5：Stage2 Transformer 基线模型

This notebook implements a Transformer Encoder baseline for A-share cross-sectional return prediction under the existing V1+V2 research framework. It uses the labeled daily panel generated by the V2 pipeline, applies walk-forward evaluation and training-window-only preprocessing, and tests whether Transformer / Multi-Head Attention can extract incremental sequence information from historical price-volume features.

本 Notebook 基于项目已有的 V1+V2 研究框架与 A 股日频标签面板，新增 Transformer Encoder 序列模型实验。在保持 V2 严格因果约束、walk-forward 样本外评估和统一 evaluator 的前提下，检验 Transformer / Multi-Head Attention 是否能够利用历史量价特征序列，提高个股未来收益预测效果。

Default task: next-trading-day return prediction using `label_1d_raw`. To run the 5-day return prediction version, set `LABEL_HORIZON = "5d"` in the configuration cell.

默认任务：预测下一交易日收益，即 `label_1d_raw`。如需运行未来 5 日收益预测版本，可在配置 Cell 中将 `LABEL_HORIZON` 改为 `"5d"`。

Before running, please confirm:

1. `PROJECT_ROOT` points to the local V2 project directory.
2. The V2 data cache and label files are available.
3. Required files such as `daily_panel_with_labels.parquet` exist locally.

运行前请确认：

1. `PROJECT_ROOT` 指向本地 V2 项目目录。
2. V2 数据缓存与标签文件已准备完成。
3. 本地存在 `daily_panel_with_labels.parquet` 等必要文件。


In [ ]:
from pathlib import Path
import os
import sys
import json
import math
import time
import copy
import logging
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Pandas:", pd.__version__)


## 1. Project Paths and Runtime Configuration  
### 项目路径与运行配置

This section defines the core project paths, prediction horizon, feature settings, training parameters, and output directories. `PROJECT_ROOT` should point to the local V2 project directory that contains the data cache, source code, and label files required by this experiment.

本节定义项目路径、预测周期、特征设置、训练参数和输出目录。`PROJECT_ROOT` 应指向本地 V2 项目目录，该目录需要包含本实验所依赖的数据缓存、源码文件和标签文件。

Default local project path:

默认本地项目路径：

```text
<LOCAL_V2_PROJECT_DIR>
```

Raw data and large intermediate files are not included in this repository. The notebook is designed to document the reproducible research workflow while keeping proprietary data outside GitHub.

本仓库不包含原始数据和大型中间文件。本 Notebook 主要用于展示可复现的研究流程，同时避免将专有数据上传至 GitHub。


In [ ]:
# =============================
# 需要重点检查这个路径
# =============================
PROJECT_ROOT = Path("<LOCAL_V2_PROJECT_DIR>")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = ARTIFACT_DIR / "figures"

for d in [ARTIFACT_DIR, REPORT_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_ROOT:", SRC_ROOT)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("Exists PROJECT_ROOT:", PROJECT_ROOT.exists())
print("Exists SRC_ROOT:", SRC_ROOT.exists())


In [ ]:
# =============================
# 导入 V2 原项目模块
# =============================
from evaluation.evaluator import evaluate_predictions, plot_evaluation
from models import ridge_baseline as v1
from models import ridge_baseline_v2 as v2

# =============================
# Transformer 实验配置
# =============================
LABEL_HORIZON = "1d"   # 可改成 "5d"，用于预测未来 5 日收益
assert LABEL_HORIZON in {"1d", "5d"}

LABEL_COL = f"label_{LABEL_HORIZON}_raw"
LABEL_TIME_COL = f"label_time_{LABEL_HORIZON}"

FEATURE_COLUMNS = list(v2.FEATURE_COLUMNS)

SEQUENCE_LENGTH = 20       # 每只股票使用过去 20 个交易观察点作为输入序列
D_MODEL = 64               # Transformer hidden dimension
N_HEADS = 4                # Multi-Head Attention heads
N_LAYERS = 2               # Transformer Encoder 层数
DROPOUT = 0.10

# 为了先跑通，默认每个 rolling window 最多抽样 120000 条训练序列
# 如果电脑性能足够，可以改成 0，表示使用全部训练样本
MAX_TRAIN_SAMPLES_PER_WINDOW = 120000
MAX_EPOCHS = 12
PATIENCE = 3
MIN_DELTA = 1e-7
BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VALIDATION_DATE_FRACTION = 0.10
RANDOM_STATE = 20260812

PREDICTIONS_PATH = ARTIFACT_DIR / f"transformer_{LABEL_HORIZON}_predictions.parquet"
WINDOWS_PATH = ARTIFACT_DIR / f"transformer_{LABEL_HORIZON}_windows.parquet"
SUMMARY_PATH = REPORT_DIR / f"transformer_{LABEL_HORIZON}_summary.md"
LOSS_FIGURE_PATH = FIGURE_DIR / f"transformer_{LABEL_HORIZON}_loss_curves.png"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# Mac MPS 偶尔对某些算子不稳定；如果遇到 device 报错，可以手动改成：
# DEVICE = torch.device("cpu")

print("LABEL_HORIZON:", LABEL_HORIZON)
print("LABEL_COL:", LABEL_COL)
print("LABEL_TIME_COL:", LABEL_TIME_COL)
print("FEATURE_COLUMNS:", FEATURE_COLUMNS)
print("N_FEATURES:", len(FEATURE_COLUMNS))
print("DEVICE:", DEVICE)
print("PREDICTIONS_PATH:", PREDICTIONS_PATH)
print("SUMMARY_PATH:", SUMMARY_PATH)


## 2. Load the V2 Panel and Build Strict T-1 Features  
### 读取 V2 panel，并构造严格 T-1 特征

This section reuses `ridge_baseline_v2.build_features` from the V2 pipeline. This keeps the Transformer input feature definition consistent with the corrected V2 Ridge baseline.

本节复用 V2 流程中的 `ridge_baseline_v2.build_features`。这样可以保证 Transformer 的输入特征口径与 V2 修正版 Ridge baseline 保持一致。

The key requirement is strict causality: all model inputs must be available before the prediction date. Therefore, the constructed features use T-1 or earlier information only.

这里的核心要求是严格因果性：所有模型输入都必须在预测日之前已经可获得。因此，构造后的特征只使用 T-1 及更早的信息。


In [ ]:
# 自动生成 V2 data_cache

# =========================
# 生成 V2 data_cache
# 必须放在 v1.load_panel() 前面运行
# =========================

from pathlib import Path
import os
import sys
import shutil
import subprocess

# 你的 V2 项目根目录
PROJECT_ROOT = Path("<LOCAL_V2_PROJECT_DIR>")

# V2 的父目录，也就是 V1+V2
PACKAGE_ROOT = PROJECT_ROOT.parent

# V2 代码默认寻找这里：
# data/daily_K
EXPECTED_DAILY_K_DIR = PACKAGE_ROOT / "data_source" / "daily_K"

# 你的原始日线 CSV 可能在这些位置
SOURCE_DAILY_CANDIDATES = [
    Path("<LOCAL_V1_DIR>/data_extracted/daily_temp3"),
    Path("<LOCAL_V2_DIR>/data_extracted/daily_temp3"),
    Path("<LOCAL_V4_DIR>/data_extracted/daily_temp3"),
    Path("<LOCAL_PROJECT_DIR>/data_extracted/daily_temp3"),
    PACKAGE_ROOT / "data_source" / "daily_K",
]

DATA_CACHE_DIR = PROJECT_ROOT / "data_cache"
DAILY_PANEL_PATH = DATA_CACHE_DIR / "daily_panel.parquet"
LABELED_PANEL_PATH = DATA_CACHE_DIR / "daily_panel_with_labels.parquet"

DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_DAILY_K_DIR.parent.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PACKAGE_ROOT:", PACKAGE_ROOT)
print("EXPECTED_DAILY_K_DIR:", EXPECTED_DAILY_K_DIR)
print("DAILY_PANEL_PATH:", DAILY_PANEL_PATH)
print("LABELED_PANEL_PATH:", LABELED_PANEL_PATH)

# 1. 确认原始 daily_K 数据目录
if not EXPECTED_DAILY_K_DIR.exists():
    source_daily_dir = None

    for p in SOURCE_DAILY_CANDIDATES:
        if p.exists() and len(list(p.glob("*.csv"))) > 0:
            source_daily_dir = p
            break

    if source_daily_dir is None:
        raise FileNotFoundError(
            "没有找到原始日线CSV目录。请确认 daily_temp3 在哪里，"
            "然后把 SOURCE_DAILY_CANDIDATES 里的路径改成你的实际路径。"
        )

    print("找到原始日线数据:", source_daily_dir)

    # 优先创建软链接，避免复制几GB数据
    try:
        os.symlink(source_daily_dir, EXPECTED_DAILY_K_DIR, target_is_directory=True)
        print("已创建软链接:")
        print(EXPECTED_DAILY_K_DIR, "->", source_daily_dir)
    except FileExistsError:
        print("daily_K 目录已存在:", EXPECTED_DAILY_K_DIR)
    except Exception as e:
        print("软链接失败，改为复制。原因:", e)
        shutil.copytree(source_daily_dir, EXPECTED_DAILY_K_DIR, dirs_exist_ok=True)
        print("已复制日线数据到:", EXPECTED_DAILY_K_DIR)

else:
    print("V2 daily_K 目录已存在:", EXPECTED_DAILY_K_DIR)

print("daily_K CSV 数量:", len(list(EXPECTED_DAILY_K_DIR.glob("*.csv"))))

# 2. 生成 daily_panel.parquet
if not DAILY_PANEL_PATH.exists():
    print("\n开始生成 daily_panel.parquet ...")

    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "data" / "loaders.py"),
        "--start-date", "2020-01-02",
        "--end-date", "2026-04-09",
        "--output", str(DAILY_PANEL_PATH),
    ]

    print("运行命令:")
    print(" ".join(cmd))

    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

else:
    print("\ndaily_panel.parquet 已存在，跳过生成。")

# 3. 生成 daily_panel_with_labels.parquet
if not LABELED_PANEL_PATH.exists():
    print("\n开始生成 daily_panel_with_labels.parquet ...")

    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "labels" / "label_builder.py"),
        "--input", str(DAILY_PANEL_PATH),
        "--output", str(LABELED_PANEL_PATH),
    ]

    print("运行命令:")
    print(" ".join(cmd))

    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

else:
    print("\ndaily_panel_with_labels.parquet 已存在，跳过生成。")

print("\n完成。现在可以运行 v1.load_panel()。")
print("存在 daily_panel_with_labels:", LABELED_PANEL_PATH.exists())

In [ ]:
# 读取 V2 panel
panel = v1.load_panel()

print("Panel shape:", panel.shape)
print("Panel columns sample:", list(panel.columns)[:30])
print("Date range:", panel["tradeDate"].min(), "to", panel["tradeDate"].max())
print("Stocks:", panel["secID"].nunique())

if LABEL_COL not in panel.columns or LABEL_TIME_COL not in panel.columns:
    raise ValueError(
        f"Panel does not contain {LABEL_COL}/{LABEL_TIME_COL}. "
        "请先运行 V2 的标签构造流程，确保 daily_panel_with_labels.parquet 已包含对应标签。"
    )

# 构造严格滞后一日的信息集特征
featured = v2.build_features(panel)

print("Featured shape:", featured.shape)
print("Feature columns:", FEATURE_COLUMNS)
print("Missing feature rows:", featured[FEATURE_COLUMNS].isna().any(axis=1).sum())
print("Label non-null rows:", featured[LABEL_COL].notna().sum())
featured[["secID", "tradeDate", LABEL_COL, LABEL_TIME_COL, "predict_time", "feature_input_end_date"] + FEATURE_COLUMNS].head()


## 3. Dataset Construction: Historical Sequences by Stock  
### Dataset：按个股构造历史序列

Each sample corresponds to one stock on one prediction date. The model uses a historical sequence of stock-level features as input and predicts the future return.

每个样本对应某只股票在某个预测日的一次预测任务。模型使用该股票历史特征序列作为输入，并预测未来收益。

```text
Past 20 trading observations -> Transformer -> Future return prediction
过去 20 个交易观察点的特征序列 -> Transformer -> 预测未来收益
```

The V2 feature builder already enforces the T-1 information rule. Therefore, even the last token in the sequence follows the strict no-look-ahead setting.

V2 的特征构造函数已经保证使用 T-1 及更早信息。因此，即使序列中的最后一个 token，也仍然满足严格无前视偏差的设定。


In [ ]:
class SequenceDataset(Dataset):
    """Per-stock rolling sequence dataset."""

    def __init__(
        self,
        frame: pd.DataFrame,
        allowed_mask: pd.Series,
        scaler: StandardScaler,
        target_scale: float,
        label_lower: float | None = None,
        label_upper: float | None = None,
        require_label: bool = True,
        max_samples: int | None = None,
        random_state: int = RANDOM_STATE,
    ) -> None:
        if not frame.index.equals(allowed_mask.index):
            raise ValueError("allowed_mask must align with frame index")

        needed = [
            "secID",
            "tradeDate",
            "predict_time",
            LABEL_TIME_COL,
            LABEL_COL,
            "feature_input_end_date",
            *FEATURE_COLUMNS,
        ]
        missing = set(needed).difference(frame.columns)
        if missing:
            raise ValueError(f"Frame is missing columns: {sorted(missing)}")

        data = frame[needed].copy()
        data["_allowed"] = allowed_mask.astype(bool).to_numpy()
        data.sort_values(["secID", "tradeDate"], kind="mergesort", inplace=True)
        data.reset_index(drop=True, inplace=True)

        raw_x = data[FEATURE_COLUMNS].to_numpy(dtype="float64")
        row_feature_ok = np.isfinite(raw_x).all(axis=1)
        scaled_x = scaler.transform(raw_x).astype("float32")

        self.groups: Dict[str, Dict[str, object]] = {}
        self.samples: List[Tuple[str, int]] = []
        self.meta: List[Dict[str, object]] = []
        self.target_scale = float(target_scale)
        self.label_lower = label_lower
        self.label_upper = label_upper
        self.require_label = require_label

        for secid, g in data.groupby("secID", sort=False, observed=True):
            idx = g.index.to_numpy()
            x = scaled_x[idx]
            ok = row_feature_ok[idx]
            allowed = g["_allowed"].to_numpy(dtype=bool)
            y_raw = g[LABEL_COL].to_numpy(dtype="float64")
            trade_dates = pd.to_datetime(g["tradeDate"]).to_numpy()
            predict_times = pd.to_datetime(g["predict_time"]).to_numpy()
            label_times = pd.to_datetime(g[LABEL_TIME_COL]).to_numpy()
            feature_end_dates = pd.to_datetime(g["feature_input_end_date"]).to_numpy()

            self.groups[str(secid)] = {
                "x": x,
                "ok": ok,
                "allowed": allowed,
                "y_raw": y_raw,
                "trade_dates": trade_dates,
                "predict_times": predict_times,
                "label_times": label_times,
                "feature_end_dates": feature_end_dates,
            }

            for i in range(SEQUENCE_LENGTH - 1, len(g)):
                if not allowed[i]:
                    continue
                if not ok[i - SEQUENCE_LENGTH + 1 : i + 1].all():
                    continue
                if require_label and not np.isfinite(y_raw[i]):
                    continue
                if pd.isna(feature_end_dates[i]):
                    continue
                # 核心因果检查：最后一个输入特征的结束日必须早于预测日
                if pd.Timestamp(feature_end_dates[i]) >= pd.Timestamp(predict_times[i]).normalize():
                    continue
                self.samples.append((str(secid), i))

        if max_samples is not None and max_samples > 0 and len(self.samples) > max_samples:
            rng = np.random.default_rng(random_state)
            chosen = rng.choice(len(self.samples), size=max_samples, replace=False)
            self.samples = [self.samples[i] for i in np.sort(chosen)]

        for secid, i in self.samples:
            g = self.groups[secid]
            self.meta.append(
                {
                    "secID": secid,
                    "tradeDate": pd.Timestamp(g["trade_dates"][i]),
                    "predict_time": pd.Timestamp(g["predict_times"][i]),
                    "label_time": pd.Timestamp(g["label_times"][i]) if not pd.isna(g["label_times"][i]) else pd.NaT,
                    "y_true": float(g["y_raw"][i]) if np.isfinite(g["y_raw"][i]) else np.nan,
                    "feature_input_end_date": pd.Timestamp(g["feature_end_dates"][i]),
                }
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, sample_id: int) -> Dict[str, torch.Tensor]:
        secid, i = self.samples[sample_id]
        g = self.groups[secid]
        seq = g["x"][i - SEQUENCE_LENGTH + 1 : i + 1]
        y = float(g["y_raw"][i]) if np.isfinite(g["y_raw"][i]) else np.nan
        if self.label_lower is not None and self.label_upper is not None and np.isfinite(y):
            y = float(np.clip(y, self.label_lower, self.label_upper))
        y_scaled = y / self.target_scale if np.isfinite(y) else np.nan
        return {
            "x": torch.from_numpy(seq),
            "y": torch.tensor(y_scaled, dtype=torch.float32),
            "sample_id": torch.tensor(sample_id, dtype=torch.long),
        }


## 4. Transformer Model  
### Transformer 模型

This section defines the Transformer Encoder sequence regressor. The model first projects raw input features into a hidden representation, adds positional encoding, processes the sequence through Transformer Encoder layers, and finally uses the last sequence token for return prediction.

本节定义 Transformer Encoder 序列回归模型。模型首先将原始输入特征映射到隐藏表示，加入位置编码，然后通过 Transformer Encoder 处理历史序列，最后使用序列最后一个 token 进行收益预测。

Model structure:

模型结构：

```text
Historical feature sequence
        ↓
Linear Projection
        ↓
Positional Encoding
        ↓
Transformer Encoder
        ↓
Last token representation
        ↓
Regression head
        ↓
Predicted future return
```

```text
历史特征序列
        ↓
线性映射
        ↓
位置编码
        ↓
Transformer Encoder
        ↓
最后一个 token 表示
        ↓
回归头
        ↓
预测未来收益
```


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class TransformerRegressor(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.input_proj = nn.Linear(input_dim, D_MODEL)
        self.pos = PositionalEncoding(D_MODEL, max_len=max(512, SEQUENCE_LENGTH + 4))
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_MODEL * 4,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head = nn.Sequential(
            nn.LayerNorm(D_MODEL),
            nn.Linear(D_MODEL, D_MODEL),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(D_MODEL, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(x)
        h = self.pos(h)
        h = self.encoder(h)
        last = h[:, -1, :]
        return self.head(last).squeeze(-1)

model_preview = TransformerRegressor(input_dim=len(FEATURE_COLUMNS))
model_preview


## 5. Training, Prediction, and Evaluation Functions  
### 训练、预测与评估函数

This section defines reusable functions for model training, out-of-sample prediction, and metric calculation. The objective function is based on regression loss, while the final evaluation focuses on cross-sectional ranking and portfolio-level performance.

本节定义模型训练、样本外预测和指标计算所需的函数。训练目标主要基于回归损失，但最终评价重点放在横截面排序能力和组合层表现上。

Key evaluation metrics include Rank IC, ICIR, direction accuracy, MSE, group returns, long-short spread, Sharpe ratio, turnover, and drawdown.

核心评价指标包括 Rank IC、ICIR、方向准确率、MSE、分组收益、多空收益、Sharpe、换手率和回撤。


In [ ]:
def mean_loss(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total = 0.0
    n = 0
    criterion = nn.MSELoss(reduction="sum")
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            pred = model(x)
            loss = criterion(pred, y)
            total += float(loss.item())
            n += len(y)
    return total / max(n, 1)


def fit_model(train_ds: SequenceDataset, val_ds: SequenceDataset, window_id: int):
    torch.manual_seed(RANDOM_STATE + window_id)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(RANDOM_STATE + window_id)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

    model = TransformerRegressor(input_dim=len(FEATURE_COLUMNS)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    best_loss = math.inf
    best_epoch = 0
    best_state = None
    stale = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total = 0.0
        n = 0
        for batch in train_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += float(loss.item()) * len(y)
            n += len(y)

        train_loss = total / max(n, 1)
        val_loss = mean_loss(model, val_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Window {window_id:02d} | Epoch {epoch:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        if val_loss < best_loss - MIN_DELTA:
            best_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"Early stop at epoch {epoch}, best_epoch={best_epoch}")
                break

    assert best_state is not None, "No model checkpoint was selected"
    model.load_state_dict(best_state)
    return model, train_losses, val_losses, best_epoch


@torch.no_grad()
def predict_model(model: nn.Module, dataset: SequenceDataset) -> pd.DataFrame:
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    model.eval()
    rows = []
    for batch in loader:
        x = batch["x"].to(DEVICE)
        pred_scaled = model(x).detach().cpu().numpy()
        sample_ids = batch["sample_id"].cpu().numpy()
        pred = pred_scaled * dataset.target_scale
        for sid, yhat in zip(sample_ids, pred):
            meta = dataset.meta[int(sid)]
            rows.append(
                {
                    "secID": meta["secID"],
                    "tradeDate": meta["tradeDate"],
                    "predict_time": meta["predict_time"],
                    "label_time": meta["label_time"],
                    "y_true": meta["y_true"],
                    "y_pred": float(yhat),
                    "feature_input_end_date": meta["feature_input_end_date"],
                }
            )
    return pd.DataFrame(rows)


def mse_metric(predictions: pd.DataFrame) -> float:
    valid = predictions["y_true"].notna() & predictions["y_pred"].notna()
    if not valid.any():
        return float("nan")
    err = predictions.loc[valid, "y_pred"].to_numpy(dtype="float64") - predictions.loc[valid, "y_true"].to_numpy(dtype="float64")
    return float(np.mean(err ** 2))


## 6. Walk-forward Training Function  
### Walk-forward 训练函数

This section implements the walk-forward evaluation procedure following the V2 evaluation contract.

本节按照 V2 的统一评价口径实现 walk-forward 样本外训练与测试流程。

```text
504 training days / 5 embargo days / 1 calendar-month test window
504 个交易日训练 / 5 个交易日 embargo / 1 个自然月测试
```

For each rolling window, the scaler, winsorization bounds, and Transformer parameters are fitted only within the training window. The test window is kept fully out-of-sample.

在每个滚动窗口中，标准化参数、winsorization 边界和 Transformer 参数都只在训练窗口内拟合。测试窗口始终保持完全样本外。


In [ ]:
def run_walk_forward(frame: pd.DataFrame):
    feature_complete = frame[FEATURE_COLUMNS].notna().all(axis=1)
    calendar = pd.DatetimeIndex(frame["tradeDate"].drop_duplicates().sort_values())
    first_usable_date = pd.Timestamp(frame.loc[feature_complete, "tradeDate"].min())
    windows = v1._monthly_windows(calendar, first_usable_date)
    if not windows:
        raise ValueError("Insufficient dates for walk-forward windows")

    print("Total windows:", len(windows))
    print("First window:", windows[0])
    print("Last window:", windows[-1])

    predictions = []
    audits = []

    for window_id, window in enumerate(windows, start=1):
        window_start = time.perf_counter()
        print("\n" + "=" * 100)
        print(f"Window {window_id}/{len(windows)} | test_month={window['test_month']}")
        print(window)

        test_date_mask = frame["tradeDate"].between(window["test_start"], window["test_end"])
        test_mask = test_date_mask & feature_complete
        test = frame.loc[test_mask]
        if test.empty:
            print("Skip empty test window")
            continue

        first_test_predict_time = test["predict_time"].min()
        train_date_mask = frame["tradeDate"].between(window["train_start"], window["train_end"])
        train_mask = (
            train_date_mask
            & feature_complete
            & frame[LABEL_COL].notna()
            & frame[LABEL_TIME_COL].notna()
            & frame[LABEL_TIME_COL].lt(first_test_predict_time)
        )
        train = frame.loc[train_mask]
        if train.empty:
            print("Skip empty train window")
            continue

        assert train["tradeDate"].nunique() == v1.TRAIN_DAYS
        assert train[LABEL_TIME_COL].max() < first_test_predict_time

        y_full = train[LABEL_COL].to_numpy(dtype="float64", copy=True)
        lower, upper = np.quantile(y_full, [0.01, 0.99])
        y_clipped = np.clip(y_full, lower, upper)
        target_scale = float(np.nanstd(y_clipped, ddof=1))
        if not np.isfinite(target_scale) or target_scale <= 0:
            target_scale = 1.0

        scaler = StandardScaler()
        scaler.fit(train[FEATURE_COLUMNS].to_numpy(dtype="float64"))

        train_dates = pd.DatetimeIndex(train["tradeDate"].drop_duplicates().sort_values())
        val_count = max(1, int(math.ceil(len(train_dates) * VALIDATION_DATE_FRACTION)))
        val_start = pd.Timestamp(train_dates[-val_count])
        fit_mask = train_mask & frame["tradeDate"].lt(val_start)
        val_mask = train_mask & frame["tradeDate"].ge(val_start)

        max_train_samples = None if MAX_TRAIN_SAMPLES_PER_WINDOW <= 0 else MAX_TRAIN_SAMPLES_PER_WINDOW

        fit_ds = SequenceDataset(
            frame,
            allowed_mask=fit_mask,
            scaler=scaler,
            target_scale=target_scale,
            label_lower=lower,
            label_upper=upper,
            require_label=True,
            max_samples=max_train_samples,
            random_state=RANDOM_STATE + window_id,
        )
        val_ds = SequenceDataset(
            frame,
            allowed_mask=val_mask,
            scaler=scaler,
            target_scale=target_scale,
            label_lower=lower,
            label_upper=upper,
            require_label=True,
            max_samples=None,
            random_state=RANDOM_STATE + window_id,
        )
        test_ds = SequenceDataset(
            frame,
            allowed_mask=test_mask,
            scaler=scaler,
            target_scale=target_scale,
            label_lower=None,
            label_upper=None,
            require_label=False,
            max_samples=None,
            random_state=RANDOM_STATE + window_id,
        )

        print("train rows full:", int(train_mask.sum()))
        print("fit sequence rows:", len(fit_ds))
        print("validation sequence rows:", len(val_ds))
        print("test sequence rows:", len(test_ds))

        if len(fit_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
            print("Skip because sequence dataset is empty")
            continue

        fit_start = time.perf_counter()
        model, train_losses, val_losses, best_epoch = fit_model(fit_ds, val_ds, window_id)
        fit_seconds = time.perf_counter() - fit_start

        predict_start = time.perf_counter()
        out = predict_model(model, test_ds)
        predict_seconds = time.perf_counter() - predict_start
        out["window_id"] = window_id
        out["train_start"] = window["train_start"]
        out["train_end"] = window["train_end"]
        predictions.append(out)

        total_seconds = time.perf_counter() - window_start
        audits.append(
            {
                **window,
                "window_id": window_id,
                "train_rows_full": int(train_mask.sum()),
                "fit_sequence_rows": len(fit_ds),
                "validation_sequence_rows": len(val_ds),
                "test_sequence_rows": len(test_ds),
                "sequence_length": SEQUENCE_LENGTH,
                "feature_columns": json.dumps(FEATURE_COLUMNS, ensure_ascii=False),
                "label_horizon": LABEL_HORIZON,
                "winsor_1pct": lower,
                "winsor_99pct": upper,
                "target_scale": target_scale,
                "best_epoch": best_epoch,
                "epochs_run": len(train_losses),
                "best_validation_loss": min(val_losses),
                "final_train_loss": train_losses[-1],
                "final_validation_loss": val_losses[-1],
                "train_loss_curve": json.dumps(train_losses),
                "validation_loss_curve": json.dumps(val_losses),
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds,
                "total_window_seconds": total_seconds,
                "device": str(DEVICE),
                "torch_version": torch.__version__,
                "d_model": D_MODEL,
                "n_heads": N_HEADS,
                "n_layers": N_LAYERS,
                "dropout": DROPOUT,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "max_train_samples_per_window": MAX_TRAIN_SAMPLES_PER_WINDOW,
            }
        )

        print(
            f"Completed window {window_id}: best_epoch={best_epoch}, "
            f"fit_seconds={fit_seconds:.1f}, predict_seconds={predict_seconds:.1f}, total={total_seconds:.1f}"
        )

        del model, fit_ds, val_ds, test_ds
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    if not predictions:
        raise RuntimeError("No prediction windows completed")

    result = pd.concat(predictions, ignore_index=True)
    result.sort_values(["tradeDate", "secID"], kind="mergesort", inplace=True)
    result.reset_index(drop=True, inplace=True)

    assert not result.duplicated(["tradeDate", "secID"]).any()
    assert result["feature_input_end_date"].lt(result["predict_time"].dt.normalize()).all()

    return result, pd.DataFrame(audits)


## 7. Run the Transformer Walk-forward Experiment  
### 正式运行 Transformer walk-forward

This cell starts the actual Transformer walk-forward training process. Runtime depends on hardware performance, number of rolling windows, model size, and the training sample cap.

这个 Cell 会正式启动 Transformer 的 walk-forward 训练流程。运行时间取决于电脑性能、滚动窗口数量、模型规模以及训练样本上限。

For the first run, the default sampling cap `MAX_TRAIN_SAMPLES_PER_WINDOW = 120000` is recommended to control runtime and memory usage.

第一次运行时，建议先保留默认抽样上限 `MAX_TRAIN_SAMPLES_PER_WINDOW = 120000`，以控制运行时间和内存占用。


In [ ]:
start_time = time.perf_counter()

predictions, audits = run_walk_forward(featured)

total_time = time.perf_counter() - start_time
print("\nAll windows completed.")
print("Predictions shape:", predictions.shape)
print("Audits shape:", audits.shape)
print("Total seconds:", total_time)

predictions.head()


## 8. Save Predictions and Window Audit Table  
### 保存预测结果和窗口审计表

This section saves the model predictions and the rolling-window audit information. Large prediction parquet files and intermediate cache files are excluded from the GitHub repository.

本节保存模型预测结果和滚动窗口审计信息。大型预测 parquet 文件和中间缓存文件不会上传至 GitHub 仓库。

The saved audit table is useful for checking the train-test split, embargo setting, sample size, and window-level training behavior.

窗口审计表可用于检查训练测试切分、embargo 设置、样本数量以及每个窗口的训练情况。


In [ ]:
predictions.to_parquet(PREDICTIONS_PATH, index=False)
audits.to_parquet(WINDOWS_PATH, index=False)

print("Saved predictions:", PREDICTIONS_PATH)
print("Saved window audit:", WINDOWS_PATH)

display(audits[[
    "window_id", "test_month", "fit_sequence_rows", "validation_sequence_rows", "test_sequence_rows",
    "best_epoch", "best_validation_loss", "final_train_loss", "final_validation_loss", "total_window_seconds"
]].head())


## 9. Evaluate with the V2 Unified Evaluator  
### 复用 V2 统一 evaluator 评估结果

This section reuses the V2 evaluator to make the Transformer baseline comparable with previous models such as Ridge, Kernel Ridge, and MLP.

本节复用 V2 的统一 evaluator，使 Transformer baseline 可以与之前的 Ridge、Kernel Ridge 和 MLP 等模型在相同口径下进行比较。

The focus is not only prediction error, but also whether the predicted scores produce useful cross-sectional ranking and implementable long-short portfolio performance.

评价重点不仅是预测误差，还包括预测分数是否能够形成有效的横截面排序，以及是否能产生具有可实现性的多空组合表现。


In [ ]:
def evaluation_input(predictions: pd.DataFrame) -> pd.DataFrame:
    return predictions[["secID", "tradeDate", "y_true", "y_pred", "predict_time", "label_time"]].copy()


eval_result = evaluate_predictions(evaluation_input(predictions), cost_bps=5)
eval_result["name"] = f"transformer_{LABEL_HORIZON}"

mse = mse_metric(predictions)

summary_table = pd.DataFrame([
    {
        "model": f"transformer_{LABEL_HORIZON}",
        "evaluated_rows": eval_result["data_quality"]["evaluated_rows"],
        "valid_days": eval_result["ic_stats"]["valid_days"],
        "rank_ic_mean": eval_result["ic_stats"]["mean"],
        "rank_ic_std": eval_result["ic_stats"]["std"],
        "icir": eval_result["ic_stats"]["icir"],
        "ic_positive_ratio": eval_result["ic_stats"]["positive_ratio"],
        "direction_accuracy": eval_result["direction_stats"]["mean_daily_accuracy"],
        "mse": mse,
        "top_annualized_return": eval_result["annualized_portfolio_stats"]["top"]["annualized_return"],
        "bottom_annualized_return": eval_result["annualized_portfolio_stats"]["bottom"]["annualized_return"],
        "spread_gross_annualized_return": eval_result["annualized_portfolio_stats"]["spread_gross"]["annualized_return"],
        "spread_net_annualized_return": eval_result["annualized_portfolio_stats"]["spread_net"]["annualized_return"],
        "net_annualized_return": eval_result["backtest_metrics"]["annualized_return"],
        "net_annualized_volatility": eval_result["backtest_metrics"]["annualized_volatility"],
        "sharpe": eval_result["backtest_metrics"]["sharpe"],
        "max_drawdown": eval_result["backtest_metrics"]["max_drawdown"],
        "mean_top_turnover": eval_result["backtest_metrics"]["mean_top_turnover"],
        "mean_bottom_turnover": eval_result["backtest_metrics"]["mean_bottom_turnover"],
    }
])

display(summary_table.T)


## 10. Generate V2 Evaluator Figures and Transformer Loss Curves  
### 生成 V2 evaluator 图和 Transformer loss 曲线

This section generates the main figures for model diagnosis and project presentation, including group returns, long-short NAV, Daily Rank IC, and Transformer training loss curves.

本节生成模型诊断和项目展示所需的主要图表，包括分组收益、多空净值、日度 Rank IC 和 Transformer 训练 loss 曲线。

These figures help evaluate whether the model has monotonic ranking power, stable out-of-sample performance, and reasonable training behavior.

这些图表有助于判断模型是否具有单调排序能力、稳定的样本外表现以及合理的训练过程。


In [ ]:
figure_paths = plot_evaluation(eval_result)
print("Evaluator figures:")
for k, v in figure_paths.items():
    print(k, v)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for row in audits.itertuples(index=False):
    train_loss = json.loads(row.train_loss_curve)
    plt.plot(range(1, len(train_loss) + 1), train_loss, alpha=0.35)
plt.title("Transformer Training Loss by Window")
plt.xlabel("Epoch")
plt.ylabel("Scaled MSE")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(LOSS_FIGURE_PATH, dpi=150)
plt.show()

print("Loss figure saved:", LOSS_FIGURE_PATH)


## 11. Write the Markdown Summary Report  
### 写出 Markdown 总结报告

This section writes a concise experiment summary report in Markdown format. The report records the model setting, test results, output files, and interpretation notes.

本节将实验结果写出为简洁的 Markdown 总结报告。报告记录模型设置、测试结果、输出文件和结果解释说明。

The report is intended for GitHub documentation and research review. Local absolute paths, raw data files, and large intermediate outputs are intentionally excluded.

该报告用于 GitHub 文档展示和研究复盘。本地绝对路径、原始数据文件和大型中间结果会被有意排除。


In [ ]:
def fmt(value, percent=False):
    if not np.isfinite(value):
        return "NaN"
    return f"{value:.2%}" if percent else f"{value:.6f}"

summary_text = f"""# Transformer Baseline Summary

## Experiment

- Model: Transformer Encoder sequence regressor
- Label horizon: `{LABEL_HORIZON}` (`{LABEL_COL}`)
- Sequence length: {SEQUENCE_LENGTH} trading observations per stock
- Features: {', '.join(FEATURE_COLUMNS)}
- Walk-forward: {v1.TRAIN_DAYS} training days / {v1.EMBARGO_DAYS} embargo days / monthly test
- Scaler and winsorization bounds fitted inside each training window only
- Training rows per window: capped at {MAX_TRAIN_SAMPLES_PER_WINDOW if MAX_TRAIN_SAMPLES_PER_WINDOW > 0 else 'all'} sequences
- Device: {DEVICE}

## Test results

| Metric | Value |
|---|---:|
| Valid evaluated rows | {eval_result['data_quality']['evaluated_rows']:,} |
| Test days | {eval_result['ic_stats']['valid_days']:,} |
| Rank IC mean | {fmt(eval_result['ic_stats']['mean'])} |
| Rank IC std | {fmt(eval_result['ic_stats']['std'])} |
| ICIR | {fmt(eval_result['ic_stats']['icir'])} |
| IC positive ratio | {fmt(eval_result['ic_stats']['positive_ratio'], True)} |
| Direction accuracy | {fmt(eval_result['direction_stats']['mean_daily_accuracy'], True)} |
| MSE | {fmt(mse)} |
| Top annualized return | {fmt(eval_result['annualized_portfolio_stats']['top']['annualized_return'], True)} |
| Bottom annualized return | {fmt(eval_result['annualized_portfolio_stats']['bottom']['annualized_return'], True)} |
| Top-Bottom gross annualized return | {fmt(eval_result['annualized_portfolio_stats']['spread_gross']['annualized_return'], True)} |
| Top-Bottom net annualized return | {fmt(eval_result['annualized_portfolio_stats']['spread_net']['annualized_return'], True)} |
| Net annualized compound return | {fmt(eval_result['backtest_metrics']['annualized_return'], True)} |
| Net annualized volatility | {fmt(eval_result['backtest_metrics']['annualized_volatility'], True)} |
| Sharpe | {fmt(eval_result['backtest_metrics']['sharpe'])} |
| Max drawdown | {fmt(eval_result['backtest_metrics']['max_drawdown'], True)} |
| Mean top turnover | {fmt(eval_result['backtest_metrics']['mean_top_turnover'], True)} |
| Mean bottom turnover | {fmt(eval_result['backtest_metrics']['mean_bottom_turnover'], True)} |

## Output files

- Predictions: `{PREDICTIONS_PATH}`
- Window audit: `{WINDOWS_PATH}`
- Summary: `{SUMMARY_PATH}`
- Loss curves: `{LOSS_FIGURE_PATH}`

## Interpretation note

This is a sequence-model baseline. It should be compared against `ridge_baseline_v2`, `mlp_baseline`, and other V2 outputs under the same evaluator before making any conclusion. A useful Transformer result should improve not only validation loss, but also out-of-sample Rank IC, net spread return, Sharpe, turnover, and drawdown under the shared evaluation contract.
"""

SUMMARY_PATH.write_text(summary_text, encoding="utf-8")
print(summary_text)
print("Saved summary:", SUMMARY_PATH)


## 12. Optional: Run the 5-day Return Prediction Version  
### 可选：运行未来 5 日收益版本

The default task in this notebook is next-trading-day return prediction, using `label_1d_raw` as the target variable. If the research objective is changed to 5-day forward return prediction, update the configuration cell as follows:

本 Notebook 的默认任务是预测下一交易日收益，目标变量为 `label_1d_raw`。如果研究目标改为预测未来 5 日收益，可以在配置 Cell 中进行如下修改：

```python
LABEL_HORIZON = "1d"
```

Change it to:

改为：

```python
LABEL_HORIZON = "5d"
```

After changing the horizon, rerun the notebook from the configuration cell onward to rebuild labels, sequences, model training, evaluation, and figures under the new prediction horizon.

修改预测周期后，需要从配置 Cell 开始重新运行后续流程，以重新生成标签、序列样本、模型训练、评估结果和图表。
